In [3]:
import pandas as pd
import numpy as np
import lfp.lfp_analysis.LFP_collection as LFP_collection
import lfp.lfp_analysis.plotting as lfplt
import lfp.lfp_analysis.event_extraction as ee
import pickle
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import ScalarFormatter
from itertools import combinations, permutations
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from importlib import reload
from collections import defaultdict

def unpickle_this(pickle_file):
    """
    Unpickles things
    Args (1):
        file_name: str, pickle filename that already exists and ends with .pkl
    Returns:
        pickled item
    """
    with open(pickle_file, "rb") as file:
        return pickle.load(file)
    


In [5]:

cagemate_lfp_json = r"C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Habituation_Dishabituation (phase 1)\lfp_data\cagemate_lfp\lfp_collection.json"

cagemate_collection = LFP_collection.LFPCollection.load_collection(cagemate_lfp_json)




In [6]:
novel_lfp_json = r"C:\Users\megha\UF Dropbox\Meghan Cum\Padilla-Coreano Lab\2024\Cum_SocialMemEphys_pilot2\Habituation_Dishabituation (phase 1)\lfp_data\novel_lfp\lfp_collection.json"

novel_collection = LFP_collection.LFPCollection.load_collection(novel_lfp_json)

In [7]:
behavior_dicts = unpickle_this(r'pilot2\habit_dishabit_phase1\behavior_dicts_from_frames.pkl')
print(behavior_dicts.keys())


for recording in cagemate_collection.recordings:
    subject = str(int(recording.name.split('_')[0])/10)
    recording_pattern = (recording.name.split('_')[0] + '_' +
                         recording.name.split('_')[1] + '_' +
                         recording.name.split('_')[2] + '_' +
                         'aggregated')
    recording.event_dict = behavior_dicts[recording_pattern]
    print(recording_pattern)
    recording.subject = subject

for recording in novel_collection.recordings:
    subject = str(int(recording.name.split('_')[0])/10)
    recording_pattern = (recording.name.split('_')[0] + '_' +
                         recording.name.split('_')[1] + '_' +
                         recording.name.split('_')[2] + '_' +
                         'aggregated')
    recording.event_dict = behavior_dicts[recording_pattern]
    print(recording_pattern)
    recording.subject = subject

dict_keys(['11_cage_p1_aggregated', '11_nov_p1_aggregated', '12_cage_p1_aggregated', '12_nov_p1_aggregated', '13_cage_p1_aggregated', '13_nov_p1_aggregated', '21_cage_p1_aggregated', '21_nov_p1_aggregated', '22_cage_p1_aggregated', '22_nov_p1_aggregated', '23_cage_p1_aggregated', '23_nov_p1_aggregated', '24_cage_p1_aggregated', '24_nov_p1_aggregated', '31_cage_p1_aggregated', '31_nov_p1_aggregated', '32_cage_p1_aggregated', '32_nov_p1_aggregated', '33_cage_p1_aggregated', '33_nov_p1_aggregated', '41_cage_p1_aggregated', '41_nov_p1_aggregated', '44_cage_p1_aggregated', '44_nov_p1_aggregated'])
11_cage_p1_aggregated
12_cage_p1_aggregated
13_cage_p1_aggregated
21_cage_p1_aggregated
22_cage_p1_aggregated
23_cage_p1_aggregated
24_cage_p1_aggregated
31_cage_p1_aggregated
32_cage_p1_aggregated
33_cage_p1_aggregated
41_cage_p1_aggregated
44_cage_p1_aggregated
11_nov_p1_aggregated
12_nov_p1_aggregated
13_nov_p1_aggregated
21_nov_p1_aggregated
22_nov_p1_aggregated
23_nov_p1_aggregated
24_nov_p1_

In [8]:
exclusion_dict = {'1.1': [],
                  '1.2': [],
                  '1.3': ['BLA'],
                  '2.1': [],
                  '2.2': ['vHPC'],
                  '2.3': ['mPFC', 'vHPC'],
                  '2.4': ['BLA', 'MD'],
                  '3.1': ['vHPC'],
                  '3.2': [],
                  '3.3': ['MD', 'NAc', 'vHPC'],
                  '4.1': [],
                  '4.4': ['NAc']}
novel_collection.exclude_regions(exclusion_dict)
cagemate_collection.exclude_regions(exclusion_dict)

In [9]:
cagemate_collection.interpolate()
novel_collection.interpolate()

In [10]:
def expected_nan_pct(excluded_regions, n_regions=5):
    """
    Calculate expected NaN percentage based on excluded regions.
    
    For power: NaN for each excluded region's channel
    For coherence/granger: NaN for any pair involving an excluded region
    """
    n_excluded = len(excluded_regions)
    
    # Power: one value per region
    power_nan = n_excluded / n_regions * 100
    
    # Coherence/Granger: n×n matrix, diagonal already NaN
    # Count pairs where at least one region is excluded
    total_directed_pairs = n_regions * (n_regions - 1)  # off-diagonal
    # Pairs involving excluded regions (from OR to excluded)
    nan_pairs = n_excluded * (n_regions - 1) * 2 - n_excluded * (n_excluded - 1)
    # Subtract double-counted excluded-to-excluded pairs
    conn_nan = nan_pairs / total_directed_pairs * 100
    
    return power_nan, conn_nan

for collection in [cagemate_collection, novel_collection]:
    for recording in collection.recordings:
        n_regions = len(recording.brain_region_dict)
        
        exp_power_nan, exp_conn_nan = expected_nan_pct(
            recording.excluded_regions, 
            n_regions=n_regions
        )
        
        actual_power_nan    = np.mean(np.isnan(recording.power))    * 100
        actual_coherence_nan = np.mean(np.isnan(recording.coherence)) * 100
        actual_granger_nan  = np.mean(np.isnan(recording.granger))  * 100
        
        # Flag subjects where actual NaN exceeds expected
        power_flag    = "⚠️" if actual_power_nan    > exp_power_nan    + 5 else "✓"
        coherence_flag = "⚠️" if actual_coherence_nan > exp_conn_nan     + 5 else "✓"
        granger_flag  = "⚠️" if actual_granger_nan  > exp_conn_nan     + 5 else "✓"
        
        print(
            f"Subject: {recording.subject}  "
            f"Excluded: {recording.excluded_regions}  \n"
            f"  Power:     actual={actual_power_nan:.1f}%  "
            f"expected={exp_power_nan:.1f}%  {power_flag}\n"
            f"  Coherence: actual={actual_coherence_nan:.1f}%  "
            f"expected={exp_conn_nan:.1f}%  {coherence_flag}\n"
            f"  Granger:   actual={actual_granger_nan:.1f}%  "
            f"expected={exp_conn_nan:.1f}%  {granger_flag}\n"
        )

Subject: 1.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 1.2  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 1.3  Excluded: ['BLA']  
  Power:     actual=20.0%  expected=20.0%  ✓
  Coherence: actual=32.0%  expected=40.0%  ✓
  Granger:   actual=32.0%  expected=40.0%  ✓

Subject: 2.1  Excluded: []  
  Power:     actual=0.0%  expected=0.0%  ✓
  Coherence: actual=0.0%  expected=0.0%  ✓
  Granger:   actual=0.0%  expected=0.0%  ✓

Subject: 2.2  Excluded: ['vHPC']  
  Power:     actual=20.0%  expected=20.0%  ✓
  Coherence: actual=32.0%  expected=40.0%  ✓
  Granger:   actual=32.0%  expected=40.0%  ✓

Subject: 2.3  Excluded: ['mPFC', 'vHPC']  
  Power:     actual=40.0%  expected=40.0%  ✓
  Coherence: actual=56.0%  expected=70.0%  ✓
  Granger:   actual=56.0%  expected=70.0%  ✓

Sub

In [13]:
import itertools
reload(ee)
regions_bidict = cagemate_collection.brain_region_dict
averages = {}
events = ['exp1 sniff','exp2 sniff', 'exp3 sniff', 'exp4 sniff', 'exp5 sniff']
baselines = ['baseline', 'baseline', 'baseline', 'baseline', 'baseline']

def create_nested_dict():
    return defaultdict(lambda: defaultdict(lambda: 
                                           defaultdict(lambda: 
                                                       defaultdict(lambda: 
                                                                   defaultdict((lambda: defaultdict(list)))))))

averages = create_nested_dict()

for recording in cagemate_collection.recordings: 
    subject = recording.subject
    for mode in ['power', 'coherence', 'granger']:
        regions = list(regions_bidict.keys())
        event_avg_dict = ee.baselined_events(recording, events=events, mode=mode, baseline=baselines)
        agent_band_dict  = ee.band_calcs(event_avg_dict, outer_dict = 'agent', freq_axis = 1)

        for event, bands in agent_band_dict.items():
            for band in bands.keys():
                #print(bands[band][0].shape) #[trials, region]
                band_average = bands[band][0]
                
                if mode == 'power':
                    for brain in regions:
                        reg_idx = cagemate_collection.brain_region_dict[brain]
                        # Store all trials for this region
                        trials_data = [band_average[trial][reg_idx] for trial in range(band_average.shape[0])]
                        averages[recording.name][subject][event][band]['power'][brain] = trials_data
                
                elif mode == 'coherence':           
                    for region1, region2 in itertools.combinations(regions, 2):
                        idx1 = cagemate_collection.brain_region_dict[region1]
                        idx2 = cagemate_collection.brain_region_dict[region2]
                        region_pair = f"{region1}_{region2}"
                        
                        # Store all trials for this region pair
                        trials_data = [band_average[trial, idx1, idx2] for trial in range(band_average.shape[0])]
                        
                        averages[recording.name][subject][event][band]['coherence'][region_pair] = trials_data
                
                elif mode == 'granger':
                    for from_region, to_region in itertools.permutations(regions, 2):
                        from_idx = cagemate_collection.brain_region_dict[from_region]
                        to_idx = cagemate_collection.brain_region_dict[to_region]
                        region_pair = f"{from_region}_to_{to_region}"
                        
                        # Store all trials for this directed region pair
                        trials_data = [band_average[trial, to_idx, from_idx] for trial in range(band_average.shape[0])]
                        
                        averages[recording.name][subject][event][band]['granger'][region_pair] = trials_data


def nested_dict_to_long_df(nested_dict):
    """
    Convert nested dictionary to long format DataFrame
    Structure: subject -> event -> band -> metric -> region_or_pair -> [trial_values]
    """
    rows = []
    for recording in nested_dict.keys():
        for subject in nested_dict[recording].keys():
            for event in nested_dict[recording][subject].keys():
                for band in nested_dict[recording][subject][event].keys():
                    for metric in nested_dict[recording][subject][event][band].keys():
                        for region_or_pair in nested_dict[recording][subject][event][band][metric].keys():
                            trial_values = nested_dict[recording][subject][event][band][metric][region_or_pair]
                            # Create a row for each trial
                            for trial_idx, value in enumerate(trial_values):
                                row = {
                                    'subject': subject,
                                    'recording': recording,
                                    'event': event,
                                    'band': band,
                                    'metric': metric,
                                    'region_or_pair': region_or_pair,
                                    'trial': trial_idx,
                                    'value': value
                                }
                                rows.append(row)
        
    return pd.DataFrame(rows)

# Convert to long dataframe
cagemate_long = nested_dict_to_long_df(averages)
cagemate_long['condition'] = 'cagemate'
cagemate_long
#df_long.to_csv("pilot2/cagemate/ultra_long_neural_data_cagemate.csv")

c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:115: RuntimeWarning: Mean of empty slice
  event_snippet = np.nanmean(event_snippet, axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:182: RuntimeWarning: Mean of empty slice
  baseline_recording = np.nanmean(np.array(baseline_averages), axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:332: RuntimeWarning: Mean of empty slice
  band_mean = np.nanmean(arr[tuple(slices)], axis=freq_axis)


,subject,recording,event,band,metric,region_or_pair,trial,value,condition
0,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,0,17.715087,cagemate
1,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,1,-29.707704,cagemate
2,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,2,34.847624,cagemate
3,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,3,-13.693997,cagemate
4,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,4,-51.831078,cagemate
...,...,...,...,...,...,...,...,...,...
148045,4.4,44_cage_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,0,-12.697081,cagemate
148046,4.4,44_cage_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,1,-7.026612,cagemate
148047,4.4,44_cage_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,2,37.135516,cagemate
148048,4.4,44_cage_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,3,-4.871029,cagemate


In [14]:
import itertools
reload(ee)
regions_bidict = novel_collection.brain_region_dict
averages = {}
events = ['exp1 sniff','exp2 sniff', 'exp3 sniff', 'exp4 sniff', 'exp5 sniff']
baselines = ['baseline', 'baseline', 'baseline', 'baseline', 'baseline']

def create_nested_dict():
    return defaultdict(lambda: defaultdict(lambda: 
                                           defaultdict(lambda: 
                                                       defaultdict(lambda: 
                                                                   defaultdict((lambda: defaultdict(list)))))))

averages = create_nested_dict()

for recording in novel_collection.recordings: 
    subject = recording.subject
    for mode in ['power', 'coherence', 'granger']:
        regions = list(regions_bidict.keys())
        event_avg_dict = ee.baselined_events(recording, events=events, mode=mode, baseline=baselines)
        agent_band_dict  = ee.band_calcs(event_avg_dict, outer_dict = 'agent', freq_axis = 1)

        for event, bands in agent_band_dict.items():
            for band in bands.keys():
                #print(bands[band][0].shape) #[trials, region]
                band_average = bands[band][0]
                
                if mode == 'power':
                    for brain in regions:
                        reg_idx = novel_collection.brain_region_dict[brain]
                        # Store all trials for this region
                        trials_data = [band_average[trial][reg_idx] for trial in range(band_average.shape[0])]
                        averages[recording.name][subject][event][band]['power'][brain] = trials_data
                
                elif mode == 'coherence':           
                    for region1, region2 in itertools.combinations(regions, 2):
                        idx1 = novel_collection.brain_region_dict[region1]
                        idx2 = novel_collection.brain_region_dict[region2]
                        region_pair = f"{region1}_{region2}"
                        
                        # Store all trials for this region pair
                        trials_data = [band_average[trial, idx1, idx2] for trial in range(band_average.shape[0])]
                        
                        averages[recording.name][subject][event][band]['coherence'][region_pair] = trials_data
                
                elif mode == 'granger':
                    for from_region, to_region in itertools.permutations(regions, 2):
                        from_idx = novel_collection.brain_region_dict[from_region]
                        to_idx = novel_collection.brain_region_dict[to_region]
                        region_pair = f"{from_region}_to_{to_region}"
                        
                        # Store all trials for this directed region pair
                        trials_data = [band_average[trial, to_idx, from_idx] for trial in range(band_average.shape[0])]
                        
                        averages[recording.name][subject][event][band]['granger'][region_pair] = trials_data


def nested_dict_to_long_df(nested_dict):
    """
    Convert nested dictionary to long format DataFrame
    Structure: subject -> event -> band -> metric -> region_or_pair -> [trial_values]
    """
    rows = []
    for recording in nested_dict.keys():
        for subject in nested_dict[recording].keys():
            for event in nested_dict[recording][subject].keys():
                for band in nested_dict[recording][subject][event].keys():
                    for metric in nested_dict[recording][subject][event][band].keys():
                        for region_or_pair in nested_dict[recording][subject][event][band][metric].keys():
                            trial_values = nested_dict[recording][subject][event][band][metric][region_or_pair]
                            # Create a row for each trial
                            for trial_idx, value in enumerate(trial_values):
                                row = {
                                    'subject': subject,
                                    'recording': recording,
                                    'event': event,
                                    'band': band,
                                    'metric': metric,
                                    'region_or_pair': region_or_pair,
                                    'trial': trial_idx,
                                    'value': value
                                }
                                rows.append(row)
        
    return pd.DataFrame(rows)

# Convert to long dataframe
novel_long = nested_dict_to_long_df(averages)
novel_long['condition'] = 'novel'
novel_long
#df_long.to_csv("pilot2/novel/ultra_long_neural_data_novel.csv")

c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:115: RuntimeWarning: Mean of empty slice
  event_snippet = np.nanmean(event_snippet, axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:182: RuntimeWarning: Mean of empty slice
  baseline_recording = np.nanmean(np.array(baseline_averages), axis=0)
c:\Users\megha\Documents\GitHub\diff_fam_social_memory_ephys\lfp\lfp_analysis\event_extraction.py:332: RuntimeWarning: Mean of empty slice
  band_mean = np.nanmean(arr[tuple(slices)], axis=freq_axis)


,subject,recording,event,band,metric,region_or_pair,trial,value,condition
0,1.1,11_nov_p1_merged.rec,exp1 sniff,Delta,power,mPFC,0,-30.920827,novel
1,1.1,11_nov_p1_merged.rec,exp1 sniff,Delta,power,mPFC,1,49.080403,novel
2,1.1,11_nov_p1_merged.rec,exp1 sniff,Delta,power,mPFC,2,-24.525345,novel
3,1.1,11_nov_p1_merged.rec,exp1 sniff,Delta,power,mPFC,3,-37.885236,novel
4,1.1,11_nov_p1_merged.rec,exp1 sniff,Delta,power,mPFC,4,-46.911570,novel
...,...,...,...,...,...,...,...,...,...
192320,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,23,-16.498980,novel
192321,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,24,25.704075,novel
192322,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,25,39.544969,novel
192323,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,26,-25.083042,novel


In [17]:
merged_df = pd.concat([cagemate_long, novel_long])
merged_df

,subject,recording,event,band,metric,region_or_pair,trial,value,condition
0,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,0,17.715087,cagemate
1,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,1,-29.707704,cagemate
2,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,2,34.847624,cagemate
3,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,3,-13.693997,cagemate
4,1.1,11_cage_p1_merged.rec,exp1 sniff,Delta,power,mPFC,4,-51.831078,cagemate
...,...,...,...,...,...,...,...,...,...
192320,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,23,-16.498980,novel
192321,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,24,25.704075,novel
192322,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,25,39.544969,novel
192323,4.4,44_nov_p1_merged.rec,exp5 sniff,High gamma,granger,vHPC_to_BLA,26,-25.083042,novel


In [12]:
import numpy as np
import pandas as pd

rows = []

for subject, events_dict in averages.items():
    for event, bands_dict in events_dict.items():
        for band, metrics_dict in bands_dict.items():
            for metric, regions_dict in metrics_dict.items():
                
                # skip gen_baseline metrics
                if 'gen_baseline' in metric:
                    continue
                
                for region_or_pair, trial_values in regions_dict.items():
                    n_trials = len(trial_values)
                    n_nan    = sum(1 for v in trial_values 
                                   if v is None or 
                                   (isinstance(v, float) and np.isnan(v)))
                    
                    rows.append({
                        'subject'        : subject,
                        'metric'         : metric,
                        'region_or_pair' : region_or_pair,
                        'n_trials'       : n_trials,
                        'n_nan'          : n_nan,
                    })

nan_df = pd.DataFrame(rows)

# Collapse across events and bands — sum trials and nans
summary = (nan_df
    .groupby(['subject', 'metric', 'region_or_pair'])
    .agg(
        total_trials = ('n_trials', 'sum'),
        total_nan    = ('n_nan',    'sum')
    )
    .assign(pct_nan = lambda x: (x.total_nan / x.total_trials * 100).round(1))
    .reset_index()
)

print(summary.to_string(index=False))

# Flag anything with any NaN
print("\nPairs with any NaN:")
print(summary[summary.total_nan > 0].to_string(index=False))

          subject     metric region_or_pair  total_trials  total_nan  pct_nan
11_CNF_merged.rec       Beta      coherence            30          0      0.0
11_CNF_merged.rec       Beta        granger            60          0      0.0
11_CNF_merged.rec       Beta          power            15          0      0.0
11_CNF_merged.rec      Delta      coherence            30          0      0.0
11_CNF_merged.rec      Delta        granger            60          0      0.0
11_CNF_merged.rec      Delta          power            15          0      0.0
11_CNF_merged.rec High gamma      coherence            30          0      0.0
11_CNF_merged.rec High gamma        granger            60          0      0.0
11_CNF_merged.rec High gamma          power            15          0      0.0
11_CNF_merged.rec  Low gamma      coherence            30          0      0.0
11_CNF_merged.rec  Low gamma        granger            60          0      0.0
11_CNF_merged.rec  Low gamma          power            15       

In [21]:
behavior_dicts.keys()

dict_keys(['11_cage_p1_aggregated', '11_nov_p1_aggregated', '12_cage_p1_aggregated', '12_nov_p1_aggregated', '13_cage_p1_aggregated', '13_nov_p1_aggregated', '21_cage_p1_aggregated', '21_nov_p1_aggregated', '22_cage_p1_aggregated', '22_nov_p1_aggregated', '23_cage_p1_aggregated', '23_nov_p1_aggregated', '24_cage_p1_aggregated', '24_nov_p1_aggregated', '31_cage_p1_aggregated', '31_nov_p1_aggregated', '32_cage_p1_aggregated', '32_nov_p1_aggregated', '33_cage_p1_aggregated', '33_nov_p1_aggregated', '41_cage_p1_aggregated', '41_nov_p1_aggregated', '44_cage_p1_aggregated', '44_nov_p1_aggregated'])

In [26]:
import re
cage_sleap = unpickle_this('pilot2/habit_dishabit_phase1/cage_sleap_dict.pkl')
novel_sleap = unpickle_this('pilot2/habit_dishabit_phase1/novel_sleap_dict.pkl')


In [27]:
cage_sleap['11_cage_p1_aggregated'].keys()

dict_keys(['exp1', 'exp2', 'exp3', 'exp4', 'exp5'])

In [30]:


def _recording_to_subject(rec_name):
    # '11_cage_p1_aggregated' → "1.1"
    num = rec_name.split('_')[0]
    return f"{num[0]}.{num[1:]}"

rows = []

for rec, events in behavior_dicts.items():
    subject      = _recording_to_subject(rec)
    condition = 'cagemate' if 'cage' in rec else 'novel'
    if condition == 'cagemate':
        sleap_dict = cage_sleap
    else:
        sleap_dict = novel_sleap
    for event, trials in events.items():
        if event in ['exp1 sniff', 'exp2 sniff', 'exp3 sniff', 'exp4 sniff', 'exp5 sniff']:
            try:
                sleap_info     = sleap_dict[rec][event.replace(" sniff", "")]
                feature_mat    = sleap_info['features_1ms']    # [frames, n_features]
                time_ms        = sleap_info['time_ms']          # [frames,] actual timestamp per frame
                feature_legend = sleap_info['feature_legend']

                for trial_idx, (start_ms, stop_ms) in enumerate(trials):
                    start_ms = int(start_ms)
                    stop_ms  = int(stop_ms)

                    mask = (time_ms >= start_ms) & (time_ms < stop_ms)
                    if not mask.any():
                        continue

                    avg_features = np.nanmean(feature_mat[mask, :], axis=0)

                    row = {
                        'subject':   subject,
                        'recording': rec.replace('aggregated', 'merged.rec'),
                        'event':     event,
                        'partner':   'B' if event == 'exp5 sniff' else 'A',
                        'trial':     trial_idx,
                        'condition':  condition,
                        'event_length': stop_ms - start_ms
                    }
                    for feat_name, feat_val in zip(feature_legend, avg_features):
                        row[feat_name] = feat_val

                    rows.append(row)
            except KeyError:
                pass

sleap_behavior_df = pd.DataFrame(rows)
sleap_behavior_df

,subject,recording,event,partner,trial,condition,event_length,distance,velocity_mouse1,velocity_mouse2,moving_angle_mouse1,moving_angle_mouse2,front_orientation_mouse1,front_orientation_mouse2,rump_orientation_mouse1,rump_orientation_mouse2,body_angle_mouse1,body_angle_mouse2
0,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.280540,0.225348,-6.406927,14.885367,2.591790,1.359749,2.320077,0.940801,2.489983,0.000000
1,1.1,11_cage_p1_merged.rec,exp1 sniff,A,1,cagemate,3464,308.278195,0.470135,0.302678,19.639775,-1.722610,2.834648,1.839122,2.396633,1.333182,1.186964,1.491628
2,1.1,11_cage_p1_merged.rec,exp1 sniff,A,2,cagemate,1594,265.470974,0.246958,0.151703,22.401964,-12.696495,3.249607,0.762406,3.096880,1.299988,0.303318,0.000000
3,1.1,11_cage_p1_merged.rec,exp1 sniff,A,3,cagemate,2356,258.733680,0.568954,0.207065,-0.092125,-4.974138,3.206650,0.760401,3.119611,1.622473,1.374884,2.781916
4,1.1,11_cage_p1_merged.rec,exp1 sniff,A,4,cagemate,884,217.622082,0.371770,0.187391,-0.382894,-8.370776,2.437932,3.819518,1.689693,4.281895,0.000000,2.767725
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1941,4.4,44_nov_p1_merged.rec,exp5 sniff,B,23,novel,1456,266.731313,0.444829,0.236916,0.058710,4.302169,0.632229,1.416772,1.562285,1.731908,0.145534,0.977712
1942,4.4,44_nov_p1_merged.rec,exp5 sniff,B,24,novel,1801,259.397107,0.095077,0.290013,2.953308,-10.992029,1.521127,0.841927,0.214731,0.353637,0.000000,3.880336
1943,4.4,44_nov_p1_merged.rec,exp5 sniff,B,25,novel,2148,379.120736,0.073544,0.077713,3.959409,-2.572169,2.169071,0.718253,3.063317,0.466019,0.843283,0.000000
1944,4.4,44_nov_p1_merged.rec,exp5 sniff,B,26,novel,970,384.237167,0.059579,0.046370,12.122784,-7.188553,1.422906,1.091060,1.128897,0.490902,3.785832,1.331866


In [31]:
glm_df = sleap_behavior_df.merge(merged_df, on=['subject', 'recording', 'event', 'trial', 'condition']) 

In [32]:
glm_df

,subject,recording,event,partner,trial,condition,event_length,distance,velocity_mouse1,velocity_mouse2,...,front_orientation_mouse1,front_orientation_mouse2,rump_orientation_mouse1,rump_orientation_mouse2,body_angle_mouse1,body_angle_mouse2,band,metric,region_or_pair,value
0,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.280540,0.225348,...,2.591790,1.359749,2.320077,0.940801,2.489983,0.0,Delta,power,mPFC,17.715087
1,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.280540,0.225348,...,2.591790,1.359749,2.320077,0.940801,2.489983,0.0,Delta,power,NAc,0.592335
2,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.280540,0.225348,...,2.591790,1.359749,2.320077,0.940801,2.489983,0.0,Delta,power,MD,-27.343484
3,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.280540,0.225348,...,2.591790,1.359749,2.320077,0.940801,2.489983,0.0,Delta,power,BLA,-37.703427
4,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.280540,0.225348,...,2.591790,1.359749,2.320077,0.940801,2.489983,0.0,Delta,power,vHPC,41.527165
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340370,4.4,44_nov_p1_merged.rec,exp5 sniff,B,27,novel,1247,275.695803,0.432228,0.602103,...,2.802888,2.247980,2.930935,2.219162,2.376548,0.0,High gamma,granger,BLA_to_vHPC,-10.797553
340371,4.4,44_nov_p1_merged.rec,exp5 sniff,B,27,novel,1247,275.695803,0.432228,0.602103,...,2.802888,2.247980,2.930935,2.219162,2.376548,0.0,High gamma,granger,vHPC_to_mPFC,40.273446
340372,4.4,44_nov_p1_merged.rec,exp5 sniff,B,27,novel,1247,275.695803,0.432228,0.602103,...,2.802888,2.247980,2.930935,2.219162,2.376548,0.0,High gamma,granger,vHPC_to_NAc,NaN
340373,4.4,44_nov_p1_merged.rec,exp5 sniff,B,27,novel,1247,275.695803,0.432228,0.602103,...,2.802888,2.247980,2.930935,2.219162,2.376548,0.0,High gamma,granger,vHPC_to_MD,4.072220


In [33]:
glm_df.to_csv('pilot2/habit_dishabit_phase1/lfp_kinematics_habit_dishabit.csv')